# Explore Garmin data, prepocess it & export to csv

## Sources:
- [Analysis of Running Activities from Garmin Watch Using Python](https://towardsdatascience.com/analysis-of-runing-activities-from-garmin-watch-using-python-99609f83314e)

## ToDo:
- decide what to do when multiple activies in same day

## Import modules

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path

# import 3rd-party modules
import pandas as pd
import numpy as np

# import local modules

## Define functions

In [6]:
# Create Function to explore the dataframes
def explore(df: pd.DataFrame) -> None:
    """
    Function to print general information about the dataframe
    """
    print("********** 1. General info of data **********")
    print(df.info())
    
    print("\n********** 2. Shape of data **********")
    print(f"Number of rows: {len(df)}")
    print(f"Number of columns: {len(df.columns)}")
    
    print("\n********** 3. Number of missing values per column **********")
    print(df.isnull().sum())
    
    print("\n********** 4. Number of duplicated values **********")
    print(df.duplicated().sum())
    
    print("\n********** 5. Number of unique values per column (NaN non included) **********")
    print(df.nunique())
    
    print("\n********** 6. Statistical info of each column **********")
    # print(df.describe(include='all').T)
    return df.describe(include='all', datetime_is_numeric=True).T

def convert_strings_to_duration(str_series):
    """
    Function to convert series of strings to durations; workaround when multiple formats in series
    """
    return pd.to_timedelta(pd.to_datetime(str_series).dt.strftime("%H:%M:%S.%f"))

def convert_durations_to_minutes(durations_series):
    # durations_series.dt.hour*60 + durations_series.dt.minute + durations_series.dt.second/60
    return durations_series.dt.total_seconds()/60

## Read data

In [3]:
# set csv path
df_path = Path("assets/data/garmin_data/Activities_20210304_20220818.csv")

# read csv into dataframe
df = pd.read_csv(df_path, parse_dates=True)

## Explore data (Exploratory data analysis)

In [4]:
df.head()

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Min Resp,Max Resp,Stress Change,Stress Start,Stress End,Avg Stress,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Cycling,2022-08-18 18:16:52,False,Schaarbeek Cycling,12.50,322,00:43:06,118,168,2.1,...,--,--,--,--,--,--,00:41:59,03:09:07,43,108
1,Cycling,2022-08-18 08:52:03,False,Ukkel Cycling,9.58,247,00:34:42,116,138,1.5,...,--,--,--,--,--,--,00:33:40,00:40:25,-12,115
2,Strength Training,2022-08-17 19:28:43,False,Strength,0.00,876,02:49:03,109,147,2.1,...,--,--,--,--,--,--,01:08:18,12:41:38,--,--
3,Cycling,2022-08-17 18:40:09,False,Vilvoorde Cycling,7.68,197,00:25:27,118,147,1.5,...,--,--,--,--,--,--,00:24:53,00:27:36,10,87
4,Cycling,2022-08-16 19:16:46,False,Kraainem Cycling,13.28,365,00:55:59,112,136,1.1,...,--,--,--,--,--,--,00:49:36,01:46:01,11,64


In [7]:
explore(df)

********** 1. General info of data **********
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 50 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Activity Type             765 non-null    object 
 1   Date                      765 non-null    object 
 2   Favorite                  765 non-null    bool   
 3   Title                     765 non-null    object 
 4   Distance                  765 non-null    object 
 5   Calories                  765 non-null    object 
 6   Time                      765 non-null    object 
 7   Avg HR                    765 non-null    int64  
 8   Max HR                    765 non-null    int64  
 9   Aerobic TE                765 non-null    object 
 10  Avg Run Cadence           765 non-null    object 
 11  Max Run Cadence           765 non-null    object 
 12  Avg Speed                 765 non-null    object 
 13  Max Speed          

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Activity Type,765,9,Cycling,371,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,765,765,2021-03-20 23:58:35,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Favorite,765,1,False,765,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Title,765,36,Vilvoorde Cycling,198,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Distance,765,414,0.00,253,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Calories,765,363,--,172,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time,765,566,00:15:34,161,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Avg HR,765.0,NaN,NaN,NaN,105.294118,27.685094,44.0,98.0,114.0,122.0,165.0
Max HR,765.0,NaN,NaN,NaN,136.592157,31.842355,58.0,124.0,144.0,157.0,199.0
Aerobic TE,765,48,0.0,169,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Clean data

In [9]:
# select columns to keep
selected_cols = [
    "Activity Type",
    "Date",
    "Distance",
    "Calories",	"Time", "Avg HR", "Max HR", "Aerobic TE", "Avg Run Cadence",
    "Max Run Cadence", "Avg Speed", "Max Speed", "Total Ascent", "Total Descent", "Avg Stride Length", 
    "Best Lap Time", "Number of Laps", "Max Temp", "Moving Time", "Elapsed Time", "Min Elevation", "Max Elevation"
    ]

df = df[selected_cols]

### ToDo: find better solution to parse duration strings
ideas: 
- coerce error to convert error to nan and then fill na with other format
- consolidate strings in 1 format

In [10]:
# convert concerned cols to date & duration time
# football_activities_df['Date'] = pd.to_datetime(pd.to_datetime(football_activities_df['Date']).dt.date)
df['Date'] = pd.to_datetime(df['Date'])
df["Date_yy-mm-dd"] = pd.to_datetime(df.Date).dt.date
df['Time'] = convert_strings_to_duration(df['Time'])
df['Elapsed Time'] = convert_strings_to_duration(df['Elapsed Time'])
df['Best Lap Time'] = df['Best Lap Time'].apply(lambda x: f"00:{x}" if x.count(":") == 1 else x) # need to add hh:
df['Best Lap Time'] = pd.to_timedelta(df['Best Lap Time'])
df['Moving Time'] = convert_strings_to_duration(df['Moving Time'])

# convert durations cols to number of minutes
duration_cols = df.select_dtypes(include=["timedelta64[ns]"]).columns

for duration_col in duration_cols:
    df[duration_col] = convert_durations_to_minutes(df[duration_col])

<ipython-input-10-16d11a75256b>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = pd.to_datetime(df['Date'])
<ipython-input-10-16d11a75256b>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Date_yy-mm-dd"] = pd.to_datetime(df.Date).dt.date
<ipython-input-10-16d11a75256b>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-d

In [11]:
# replace dummy value "--" by 0 (toDo: check impact of this replacement for other activities)
df["Calories"].replace({'--':'0'}, inplace=True)

# replace dummy value "--" by nan
df.replace({'--':np.nan}, inplace=True)

# remove comma (to indicate thousands)
df["Calories"] = df["Calories"].str.replace(',', "")
df["Distance"] = df["Distance"].str.replace(',', "")

/Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/pandas/core/series.py:4509: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().replace(
/Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/pandas/core/frame.py:4524: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().replace(
<ipython-input-11-1154e874e224>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.ht

In [ ]:
for object_col in object_cols:
    df[object_col] = df[object_col].str.replace(',', "")

In [12]:
df["Calories"]

0      322
1      247
2      876
3      197
4      365
      ... 
760      0
761      1
762      7
763      0
764    411
Name: Calories, Length: 765, dtype: object

In [57]:
# convert object cols to float
## select only object cols
object_cols = df.select_dtypes(include='object').columns.to_list()

## remove title from object cols
object_cols.remove("Activity Type")
object_cols.remove("Avg Speed") # toDo: convert swimming pace (e.g. 1:58 /100m to kph)
object_cols.remove("Max Speed") # toDo: convert swimming pace (e.g. 1:58 /100m to kph)
object_cols.remove("Date_yy-mm-dd")

## convert object cols to float
df[object_cols] = df[object_cols].astype("float")

## inspect cols info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Activity Type      765 non-null    object        
 1   Date               765 non-null    datetime64[ns]
 2   Distance           765 non-null    float64       
 3   Calories           765 non-null    float64       
 4   Time               765 non-null    float64       
 5   Avg HR             765 non-null    int64         
 6   Max HR             765 non-null    int64         
 7   Aerobic TE         764 non-null    float64       
 8   Avg Run Cadence    764 non-null    float64       
 9   Max Run Cadence    764 non-null    float64       
 10  Avg Speed          513 non-null    object        
 11  Max Speed          513 non-null    object        
 12  Total Ascent       481 non-null    float64       
 13  Total Descent      480 non-null    float64       
 14  Avg Stride

In [58]:
df.head()

,Activity Type,Date,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,Avg Run Cadence,Max Run Cadence,...,Total Descent,Avg Stride Length,Best Lap Time,Number of Laps,Max Temp,Moving Time,Elapsed Time,Min Elevation,Max Elevation,Date_yy-mm-dd
0,Cycling,2022-08-18 18:16:52,12.50,322.0,43.100000,118,168,2.1,0.0,0.0,...,106.0,0.0,8.744267,3,0.0,41.983333,189.116667,43.0,108.0,2022-08-18
1,Cycling,2022-08-18 08:52:03,9.58,247.0,34.700000,116,138,1.5,0.0,0.0,...,69.0,0.0,13.636750,2,0.0,33.666667,40.416667,-12.0,115.0,2022-08-18
2,Strength Training,2022-08-17 19:28:43,0.00,876.0,169.050000,109,147,2.1,0.0,0.0,...,NaN,0.0,169.051500,1,0.0,68.300000,761.633333,NaN,NaN,2022-08-17
3,Cycling,2022-08-17 18:40:09,7.68,197.0,25.450000,118,147,1.5,0.0,0.0,...,105.0,0.0,8.838833,2,0.0,24.883333,27.600000,10.0,87.0,2022-08-17
4,Cycling,2022-08-16 19:16:46,13.28,365.0,55.983333,112,136,1.1,0.0,0.0,...,94.0,0.0,13.459233,3,0.0,49.600000,106.016667,11.0,64.0,2022-08-16


In [59]:
df[["Date"]].nunique()

Date    765
dtype: int64

## Group activities by date

In [64]:
# select columns
selected_cols = ["Date_yy-mm-dd", "Activity Type", "Distance", "Calories", "Time", "Avg HR", "Max HR"]

# group dataframe by date and activity Type
grouped_df_group = df[selected_cols].groupby(["Date_yy-mm-dd", "Activity Type"])
# grouped_by_date_df_group.groups

# aggregate values
grouped_df = grouped_df_group.agg({
    'Distance' : 'sum', 
    'Calories' : 'sum', 
    'Time' : 'sum', 
    'Avg HR' : 'mean',
    'Max HR' : 'mean'
    })

# reset index to have Date_yy-mm-dd col as regular column
grouped_df.reset_index(inplace=True)

# now, we can convert Date_yy-mm-dd column to datetime
grouped_df["Date_yy-mm-dd"] = pd.to_datetime(grouped_df["Date_yy-mm-dd"])
grouped_df.head()

In [81]:
# get date range of dataframe
date_start = df[["Date"]].min().dt.date.values[0]
date_end = df[["Date"]].max().dt.date.values[0]
print(date_start)
print(date_end)

# get days between range date and make a dataframe from these days
range_date = pd.date_range(start=date_start, end=date_end, freq ='D')
range_date_df = pd.DataFrame(range_date, columns = ['Date_yy-mm-dd'])

# add column to dataframe: day name
range_date_df["day_name"] = pd.to_datetime(range_date_df["Date_yy-mm-dd"]).dt.day_name()
range_date_df

2021-03-04
2022-08-18


In [98]:
# merge dataframe with all days from date range and dataframe with aggregated activities values
everyday_grouped_df = pd.merge(range_date_df, grouped_df, how="left", on="Date_yy-mm-dd")
everyday_grouped_df.head(20)

,Date_yy-mm-dd,day_name,Activity Type,Distance,Calories,Time,Avg HR,Max HR
0,2021-03-04,Thursday,Strength Training,0.00,411.0,62.933333,119.0,153.0
1,2021-03-05,Friday,Breathwork,0.00,0.0,5.841667,59.0,73.0
2,2021-03-05,Friday,Pilates,0.00,7.0,5.665000,75.0,92.0
3,2021-03-05,Friday,Yoga,0.00,1.0,2.826667,75.0,86.0
4,2021-03-06,Saturday,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
726,2022-08-16,Tuesday,Cycling,26.86,684.0,101.900000,114.0,140.0
727,2022-08-16,Tuesday,Other,1.27,300.0,61.500000,104.0,124.0
728,2022-08-17,Wednesday,Cycling,7.68,197.0,25.450000,118.0,147.0
729,2022-08-17,Wednesday,Strength Training,0.00,876.0,169.050000,109.0,147.0


## Export dataframes as csv

In [60]:
# set output csv path
out_csv_path = df_path.parent / f"{df_path.stem}_cleaned.csv"
df.to_csv(out_csv_path, index=False)

In [65]:
# set output csv path
out_csv_path = df_path.parent / f"{df_path.stem}_cleaned_grouped.csv"
grouped_df.to_csv(out_csv_path, index=False)

In [101]:
# set output csv path
out_csv_path = df_path.parent / f"{df_path.stem}_cleaned_grouped_all_days.csv"
grouped_df.to_csv(out_csv_path, index=False)